<a href="https://colab.research.google.com/github/predpoke/SuperSecondProject/blob/KAN-4-image-analysis-model/SuperCaptionModel2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

from google.colab import drive
drive.mount('/content/drive')


PROJECT_DIR = "/content/drive/MyDrive/officehome_caption_project"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
FAISS_DIR = os.path.join(PROJECT_DIR, "faiss_index")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

for path in [PROJECT_DIR, DATA_DIR, FAISS_DIR, MODEL_DIR, OUTPUT_DIR, LOG_DIR]:
  os.makedirs(path, exist_ok=True)
print("프로젝트 폴더 생성 완료")
print(PROJECT_DIR)

Mounted at /content/drive
프로젝트 폴더 생성 완료
/content/drive/MyDrive/officehome_caption_project


In [14]:
#Drive 폴더로 캡션 복사


from google.colab import files
uploaded = files.upload()

import shutil
src = "/content/office_home_enriched_metadata.csv"
dst = "/content/drive/MyDrive/officehome_caption_project/data/office_home_enriched_metadata.csv"

shutil.copy(src, dst)

Saving office_home_image_caption_metadata_pilot100.csv to office_home_image_caption_metadata_pilot100 (1).csv


'/content/drive/MyDrive/officehome_caption_project/data/office_home_enriched_metadata.csv'

In [24]:
csv_path = "/content/drive/MyDrive/officehome_caption_project/data/office_home_enriched_metadata.csv"

In [25]:
!pip install -q sentence-transformers faiss-cpu
import pandas as pd

df = pd.read_csv(csv_path)

df.head()

,dataset,row_idx,source_filepath,image_url,domain,object,price_usd,size,caption
0,Voxel51/Office-Home,0,data/data_0/00001.jpg,https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00001.jpg,Real World,Alarm_Clock,53.63,standard,"A catalog caption identifies an Alarm Clock in a real-world photograph, captured with a simple front-facing composition, against a plain or low-detail background, and useful for a marketplace-style item listing."
1,Voxel51/Office-Home,1,data/data_0/00002.jpg,https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00002.jpg,Real World,Alarm_Clock,477.95,large,"This Alarm Clock reference uses a real-world photograph; it is captured with a simple front-facing composition, with a background that does not dominate the item, and ready for visual search, tagging, or inventory matching."
2,Voxel51/Office-Home,2,data/data_0/00003.jpg,https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00003.jpg,Real World,Alarm_Clock,161.73,XS,"The Real World image contains an Alarm Clock, captured with a simple front-facing composition, using a practical reference-image style, and suited to a searchable product dataset."
3,Voxel51/Office-Home,3,data/data_0/00004.jpg,https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00004.jpg,Real World,Alarm_Clock,366.93,oversized,"A category-focused real-world photograph of an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and designed for object-aware metadata enrichment."
4,Voxel51/Office-Home,4,data/data_0/00005.jpg,https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_0/00005.jpg,Real World,Alarm_Clock,137.78,M,"An Alarm Clock in a real-world photograph is captured with a simple front-facing composition, against a plain or low-detail background, and suitable for an office supply or home goods catalog."


In [29]:
#검색용 문장 만들기

def make_search_text(row):
  object_name = str(row.get("object", "")).replace("_", " ")
  domain = str(row.get("domain", ""))
  size = str(row.get("size", ""))
  price = str(row.get("price_usd", ""))
  caption = str(row.get("caption", ""))


  text = (
      f"object : {object_name} . "
      f"Domain : {domain}."
      f"Size : {size}"
      f"Price : {price} USD."
      f"Caption : {caption}. "
  )

  return text


df["search_text"] = df.apply(make_search_text, axis =1)
df[["object", "caption", "search_text"]].head()


,object,caption,search_text
0,Alarm_Clock,"A catalog caption identifies an Alarm Clock in a real-world photograph, captured with a simple front-facing composition, against a plain or low-detail background, and useful for a marketplace-style item listing.","object : Alarm Clock . Domain : Real World.Size : standardPrice : 53.63 USD.Caption : A catalog caption identifies an Alarm Clock in a real-world photograph, captured with a simple front-facing composition, against a plain or low-detail background, and useful for a marketplace-style item listing.."
1,Alarm_Clock,"This Alarm Clock reference uses a real-world photograph; it is captured with a simple front-facing composition, with a background that does not dominate the item, and ready for visual search, tagging, or inventory matching.","object : Alarm Clock . Domain : Real World.Size : largePrice : 477.95 USD.Caption : This Alarm Clock reference uses a real-world photograph; it is captured with a simple front-facing composition, with a background that does not dominate the item, and ready for visual search, tagging, or inventory matching.."
2,Alarm_Clock,"The Real World image contains an Alarm Clock, captured with a simple front-facing composition, using a practical reference-image style, and suited to a searchable product dataset.","object : Alarm Clock . Domain : Real World.Size : XSPrice : 161.73 USD.Caption : The Real World image contains an Alarm Clock, captured with a simple front-facing composition, using a practical reference-image style, and suited to a searchable product dataset.."
3,Alarm_Clock,"A category-focused real-world photograph of an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and designed for object-aware metadata enrichment.","object : Alarm Clock . Domain : Real World.Size : oversizedPrice : 366.93 USD.Caption : A category-focused real-world photograph of an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and designed for object-aware metadata enrichment.."
4,Alarm_Clock,"An Alarm Clock in a real-world photograph is captured with a simple front-facing composition, against a plain or low-detail background, and suitable for an office supply or home goods catalog.","object : Alarm Clock . Domain : Real World.Size : MPrice : 137.78 USD.Caption : An Alarm Clock in a real-world photograph is captured with a simple front-facing composition, against a plain or low-detail background, and suitable for an office supply or home goods catalog.."


In [30]:
#


from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(embedding_model_name)

texts = df["search_text"].fillna("").tolist()

embeddings= model.encode(
    texts,
    batch_size = 64,
    show_progress_bar = True,
    convert_to_tensor = True,
    normalize_embeddings=True
)

print("임베딩 shape", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/244 [00:00<?, ?it/s]

임베딩 shape torch.Size([15588, 384])


In [32]:
!pip install -q faiss-cpu

import faiss
import numpy as np
import torch

if isinstance(embeddings, torch.Tensor):
  embeddings_np = embeddings.detach().cpu().numpy()
else :
  embeddings_np = embeddings

embeddings_np = embeddings_np.astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

In [39]:
import json

faiss_index_path = os.path.join(FAISS_DIR , "officehome_caption.index")
metadata_path = os.path.join(DATA_DIR, "officehome_search_metadata.csv")
config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

faiss.write_index(index, faiss_index_path)

df.to_csv(metadata_path, index=False)

config = {
    "embedding_model_name": embedding_model_name,
    "embedding_dim ": int(dimension),
    "num_vectors" : int(index.ntotal),
    "metadata_path" : metadata_path,
    "faiss_index_path" : faiss_index_path
}

with open(config_path, "w", encoding="utf-8") as f:
  json.dump(config, f, ensure_ascii=False, indent=2)


In [40]:
def search_caption(query, top_k =5):
  query_vec = model.encode(
      [query],
      convert_to_numpy=True,
      normalize_embeddings=True
  ).astype("float32")


  scores, indices = index.search(query_vec, top_k)


  results = df.iloc[indices[0]].copy()
  results["score"] = scores[0]

  cols = ["score", "object", "size", "price_usd", "caption", "image_url", "source_filepath"]
  available_cols = [c for c in cols if c in results. columns]

  return results[available_cols]


search_caption("black moniter with a stand", top_k =5)

,score,object,size,price_usd,caption,image_url,source_filepath
13550,0.431705,Monitor,oversized,495.23,"This clip-art style illustration presents a Monitor as the primary object, presented with emphasis on the item silhouette, with a background that does not dominate the item, and useful as a short ecommerce-style description.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_67/00031-199.jpg,data/data_67/00031-199.jpg
2405,0.430854,Monitor,compact,15.33,"This real-world photograph presents a Monitor as the primary object, captured with a simple front-facing composition, with a composition suitable for search indexing, and ready for visual search, tagging, or inventory matching.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_12/00025-34.jpg,data/data_12/00025-34.jpg
6836,0.430760,Monitor,oversized,453.03,"This product listing photo shows a Monitor, with a compact scene around the object; the item is presented with emphasis on the item silhouette and written for a compact retail caption field.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_34/00060-70.jpg,data/data_34/00060-70.jpg
10322,0.421337,Monitor,XL,253.95,"The Art image contains a Monitor, captured with a simple front-facing composition, with a composition suitable for search indexing, and suitable for an office supply or home goods catalog.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_51/00023-157.jpg,data/data_51/00023-157.jpg
10316,0.419330,Monitor,compact,85.69,"This artistic rendering shows a Monitor, with the item separated from visual distractions; the item is presented as the main subject and suited to a searchable product dataset.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_51/00017-164.jpg,data/data_51/00017-164.jpg


In [41]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)




In [42]:
print("전체 데이터 수:", len(df))
print("컬럼 목록:", df.columns.tolist())

print("\ncaption 결측치 개수:", df["caption"].isna().sum())
print("caption 빈 문자열 개수:", (df["caption"].fillna("").str.strip() == "").sum())

df[["object", "size", "price_usd", "caption"]].head(10)

전체 데이터 수: 15588
컬럼 목록: ['dataset', 'row_idx', 'source_filepath', 'image_url', 'domain', 'object', 'price_usd', 'size', 'caption', 'search_text']

caption 결측치 개수: 0
caption 빈 문자열 개수: 0


,object,size,price_usd,caption
0,Alarm_Clock,standard,53.63,"A catalog caption identifies an Alarm Clock in a real-world photograph, captured with a simple front-facing composition, against a plain or low-detail background, and useful for a marketplace-style item listing."
1,Alarm_Clock,large,477.95,"This Alarm Clock reference uses a real-world photograph; it is captured with a simple front-facing composition, with a background that does not dominate the item, and ready for visual search, tagging, or inventory matching."
2,Alarm_Clock,XS,161.73,"The Real World image contains an Alarm Clock, captured with a simple front-facing composition, using a practical reference-image style, and suited to a searchable product dataset."
3,Alarm_Clock,oversized,366.93,"A category-focused real-world photograph of an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and designed for object-aware metadata enrichment."
4,Alarm_Clock,M,137.78,"An Alarm Clock in a real-world photograph is captured with a simple front-facing composition, against a plain or low-detail background, and suitable for an office supply or home goods catalog."
5,Alarm_Clock,XL,313.96,"This real-world photograph presents an Alarm Clock as the primary object, captured with a simple front-facing composition, with a background that does not dominate the item, and appropriate for product discovery and category browsing."
6,Alarm_Clock,XL,210.43,"A labeled Alarm Clock image uses a real-world photograph; it is captured with a simple front-facing composition, using a practical reference-image style, and written for a compact retail caption field."
7,Alarm_Clock,oversized,493.68,"In the Real World domain, an Alarm Clock is captured with a simple front-facing composition, with the surrounding area kept secondary, and prepared for lightweight catalog annotation."
8,Alarm_Clock,oversized,269.10,"Office-Home presents an Alarm Clock as a real-world photograph; it is captured with a simple front-facing composition, against a plain or low-detail background, and ready for visual search, tagging, or inventory matching."
9,Alarm_Clock,standard,179.47,"A real-world photograph of an Alarm Clock, with a background that does not dominate the item; it is captured with a simple front-facing composition and prepared as synthetic commerce metadata."


In [43]:
## 여기서부터 gradio 작업 시작



!pip install -q gradio


PROJECT_DIR = "/content/drive/MyDrive/officehome_caption_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
FAISS_DIR =os.path.join(PROJECT_DIR, 'faiss_index')

faiss_index_path = os.path.join(FAISS_DIR, "officehome_caption.index")
metadata_path = os.path.join(DATA_DIR, "officehome_search_metadata.csv")
config_path = os.path.join(FAISS_DIR, "officehome_caption_config.json")

#config 불러오기
with open(config_path, "r", encoding="utf-8") as f:
  config = json.load(f)

embedding_model_name = config["embedding_model_name"]


#모델, metadata, FAISS index 불러오기

model = SentenceTransformer(embedding_model_name)
df = pd.read_csv(metadata_path)
index = faiss.read_index(faiss_index_path)


#혹시 이름 바꾸면 이거 쓰기
# faiss_index_path = os.path.join(FAISS_DIR, "네가_저장한_index_파일명.index")
# metadata_path = os.path.join(DATA_DIR, "네가_저장한_metadata_파일명.csv")
# config_path = os.path.join(FAISS_DIR, "네가_저장한_config_파일명.json")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [27]:
print(config)
print(config.keys())

{'embedding_model_name ': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dim ': 384, 'num_vectors': 15588, 'metadata_path': '/content/drive/MyDrive/officehome_caption_project/data/officehome_search_metadata.csv', 'faiss_index_path': '/content/drive/MyDrive/officehome_caption_project/faiss_index/officehome_caption.index'}
dict_keys(['embedding_model_name ', 'embedding_dim ', 'num_vectors', 'metadata_path', 'faiss_index_path'])


In [48]:
def search_caption_gradio(query, top_k) :
  if query is None or query.strip() == "":
    return pd.DataFrame({"message" : ["검색어를 입력해주세요."]})

  top_k = int(top_k)

  query_vec = model.encode(
      [query],
      convert_to_numpy=True,
      normalize_embeddings=True
  ).astype("float32")

  scores, indices = index.search(query_vec, top_k)

  results = df.iloc[indices[0]].copy()
  results["score"] = scores[0]

  cols =[
      "score",
      "object",
      "domain",
      "size",
      "price_usd",
      "caption",
      "image_url",
      "source_filepath"
  ]


  available_cols = [c for c in cols if c in results.columns]

  results = results[available_cols]


  #score 반올림 보기 좋으라고
  if "score" in results.columns :
    results["score"] = results["score"].round(4)

    return results



In [49]:
search_caption_gradio("black moniter with a stand ", 5)


,score,object,domain,size,price_usd,caption,image_url,source_filepath
13550,0.4317,Monitor,Clipart,oversized,495.23,"This clip-art style illustration presents a Monitor as the primary object, presented with emphasis on the item silhouette, with a background that does not dominate the item, and useful as a short ecommerce-style description.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_67/00031-199.jpg,data/data_67/00031-199.jpg
2405,0.4309,Monitor,Real World,compact,15.33,"This real-world photograph presents a Monitor as the primary object, captured with a simple front-facing composition, with a composition suitable for search indexing, and ready for visual search, tagging, or inventory matching.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_12/00025-34.jpg,data/data_12/00025-34.jpg
6836,0.4308,Monitor,Product,oversized,453.03,"This product listing photo shows a Monitor, with a compact scene around the object; the item is presented with emphasis on the item silhouette and written for a compact retail caption field.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_34/00060-70.jpg,data/data_34/00060-70.jpg
10322,0.4213,Monitor,Art,XL,253.95,"The Art image contains a Monitor, captured with a simple front-facing composition, with a composition suitable for search indexing, and suitable for an office supply or home goods catalog.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_51/00023-157.jpg,data/data_51/00023-157.jpg
10316,0.4193,Monitor,Art,compact,85.69,"This artistic rendering shows a Monitor, with the item separated from visual distractions; the item is presented as the main subject and suited to a searchable product dataset.",https://huggingface.co/datasets/Voxel51/Office-Home/resolve/main/data/data_51/00017-164.jpg,data/data_51/00017-164.jpg


In [51]:
import gradio as gr

with gr.Blocks(title="OfficeHome Caption Search") as demo :
  gr.Markdown(
      """
      자연어 검색어를 입력하면,
      SentenceTransformer + FAISS를 이용해 가장 유사한 이미지 캡션 데이털르 검색합니다
      """
  )


  with gr.Row():
    query_input = gr.Textbox(
        label="검색어",
        placeholder="예 : black moniter with a stand / 투명한 물병 / small alarm"
    )

    top_k_input = gr.Slider(
        minimum=1,
        maximum=20,
        value=5,
        step=1,
        label="검색 결과 개수"
    )

  search_button = gr.Button("검색하기")

  result_table = gr.Dataframe(
     label = "검색결과",
     wrap=True
 )

  search_button.click(
      fn =search_caption_gradio,
      inputs=[query_input, top_k_input],
      outputs=result_table
)


demo.launch(share=True)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0589e15c9fd814871.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [54]:
search_caption_html("black monitor with a stand", 5)

NameError: name 'search_caption_html' is not defined

In [55]:


import gradio as gr

with gr.Blocks(title="OfficeHome Caption Search") as demo:
    gr.Markdown(
        """
        # OfficeHome Caption Search

        자연어 검색어를 입력하면,
        FAISS가 가장 유사한 이미지 캡션 데이터를 찾아 이미지 카드로 보여줍니다.
        """
    )

    with gr.Row():
        query_input = gr.Textbox(
            label="검색어",
            placeholder="예: black monitor with a stand / 투명한 물병 / small alarm clock",
            lines=2
        )

        top_k_input = gr.Slider(
            minimum=1,
            maximum=12,
            value=5,
            step=1,
            label="검색 결과 개수"
        )

    search_button = gr.Button("검색하기")

    result_html = gr.HTML(label="검색 결과")

    search_button.click(
        fn=search_caption_gradio,
        inputs=[query_input, top_k_input],
        outputs=result_html
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f340a6b9f8eb63dfa7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
